In [3]:
from typing import List, Any
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from data_loader import load_all_documents

class EmbeddingPipeline:
    """
    This class handles document chunking and embedding them into vectors
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", chunk_size: int = 1000, chunk_overlap: int = 200) -> None:
        """  
            This constructor loads the given model and initialize the variables
            Args:
                model_name: str
                chunk_size: int
                chunk_overlap: int
        """
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self) -> None:
        """
        Load the sentence transformer model
        """
        try:
            print(f"[INFO] Loading Embedded model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"[INFO] Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"[DEBUG] Error during loading model {self.model_name}: {e}")
            raise
    
    def chunk_documents(self, documents: List[Any]) -> List[Any]:
        """  
        This function split the documents into small chunks
        """
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = self.chunk_size,
            chunk_overlap = self.chunk_overlap,
            length_function = len,
            separators = ["\n\n", "\n", " ", ""]
        )
        chunks = [text_splitter.split_documents(doc) for doc in documents]

        # Count documents
        docs_cnt = 0
        for doc in documents:
            docs_cnt += len(doc)
        
        # Count total chunks
        chunks_cnt = 0
        for chunk in chunks:
            chunks_cnt += len(chunk)

        print(f"[INFO] Splitted {docs_cnt} documents into {chunks_cnt} chunks")
        
        return chunks
    
    def embed_chunks(self, chunks_list: List[Any]) -> np.ndarray:
        """
        Generate embeddings for a list of chunks

        Args:
            List of texts to embed
        Returns: 
            numpy array of embeddings with shape
        """
        if not self.model_name:
            raise ValueError("Model not loaded yet!")
        
        # Create a list with page_content of a chunk
        texts = []
        for chunks in chunks_list:
            texts.extend([chunk.page_content for chunk in chunks])
 
        print(f"[INFO] Generating embeddings for {len(texts)} chunks...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"[INFO] Embeddings shape: {embeddings.shape}")
        return embeddings

docs = load_all_documents("..\data")
emb_pipe = EmbeddingPipeline()

# docs -> List(Documents)
chunks = emb_pipe.chunk_documents(docs)
# metadatas = []
# for chunk in chunks:
#     metadatas.extend([{"text": doc.page_content, "source":doc.metadata['source_file'], "page":doc.metadata['page']} for doc in chunk])
# print(metadatas)

[DEBUG] Data Path: ..\data
[DEBUG] Found 2 pdf files: ['..\\data\\pdf\\Money Market.pdf', '..\\data\\pdf\\Stock Market.pdf']
[DEBUG] Processing pdf file: ..\data\pdf\Money Market.pdf
[DEBUG] Loaded 18 pdf files from ..\data\pdf\Money Market.pdf
[DEBUG] Processing pdf file: ..\data\pdf\Stock Market.pdf
[DEBUG] Loaded 15 pdf files from ..\data\pdf\Stock Market.pdf
[DEBUG] Found 2 text files: ['..\\data\\txt\\ml_intro.txt', '..\\data\\txt\\python_intro.txt']
[DEBUG] Processing txt file: ..\data\txt\ml_intro.txt
[DEBUG] Loaded 1 text docs from ..\data\txt\ml_intro.txt
[DEBUG] Processing txt file: ..\data\txt\python_intro.txt
[DEBUG] Loaded 1 text docs from ..\data\txt\python_intro.txt
[DEBUG] Found 1 csv files: ['..\\data\\csv\\Sold Records.csv']
[DEBUG] Processing csv file: ..\data\csv\Sold Records.csv
[DEBUG] Loaded 2 csv docs from ..\data\csv\Sold Records.csv
[DEBUG] Found 0 Excel files: []
[DEBUG] Found 0 Word files: []
[DEBUG] Found 0 JSON files: []
[DEBUG] Total loaded documents; 5
[

In [11]:
metadatas = []
for chunk in chunks:
    metadatas.extend([{"text": doc.page_content, "source":doc.metadata.get('source_file'), "page_no":doc.metadata.get('page')} for doc in chunk])
print(metadatas)

[{'text': 'Money Market in India \n \n \nProf. Shailendra Singh Bhadouria \nProfessor & Ex Head, Department of Commerce \nEx Dean, Faculty of Commerce & Management \nIndira Gandhi National Tribal University \n(A Central University) \nAmarkantak (MP) 484887 \n \n \n \nMeaning of Money Market: \nMoney market is a market for short -term funds. We define the short -term as a period of 364 days or \nless. In other words, the borrowing and repayment take place in 364 days or less. The manufacturers \nneed two types of finance: finance to meet daily expenses like purchase of raw material, payment of \nwages, excise duty, electricity charges etc., and finance to meet capital expenditure like purchase of \nmachinery, installation of pollution control equipment etc. \nThe first category of finance is invested in the production process for a short-period of time. The market \nwhere such short-time finance is borrowed and lent is called ‘money market’. Almost every concern in', 'source': 'Money Ma